# TradingV1: Demo Relations in PostgreSQL

This notebook demonstrates CRUD operations on a relational schema in PostgreSQL (`postgresql-shaped-71623`) for the `seanstrader` app. The schema includes:
- **traders**: Trader information.
- **profiles**: Trader profiles (one-to-one).
- **trades**: Trades by traders (one-to-many).
- **tags**: Trade tags.
- **trade_tags**: Junction table for trades ↔ tags (many-to-many).

We'll use `DatabaseHandler`, `ConfigManager`, and `Logging` to perform operations.

**Setup**:
- Database: Heroku PostgreSQL (`postgresql-shaped-71623`)
- Config: `../config/demo_config.yaml` (overridden by `DATABASE_URL`)
- Logs: Written to `../logs/demo_relations_notebook.log`

In [1]:
import sys
import os

# Add project root to Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(project_root)

from sqlalchemy.sql import text
from helper.config_manager import ConfigManager
from helper.Logger import Logging
from helper.database import DatabaseHandler
from helper.timezone import get_formatted_timestamp

# Check for DATABASE_URL in environment
if 'DATABASE_URL' not in os.environ:
    print("WARNING: DATABASE_URL not set in environment. Falling back to config/demo_config.yaml.")
    print("For Heroku database, set DATABASE_URL environment variable or use the line below.")
    # Set Heroku DATABASE_URL for testing
    os.environ['DATABASE_URL'] = 'postgres://u64g8eocr7nhul:p3b8ecfc9a637f11fb55f26fcf499d36f5f6a5c90a59f1c99a40fce32e4b549b7@ca932070ke6bv1.cluster-czrs8kj4isg7.us-east-1.rds.amazonaws.com:5432/da1co8rkl4afh'
else:
    print(f"Using DATABASE_URL from environment: {os.environ.get('DATABASE_URL')}")

# Initialize helpers
config = ConfigManager("../config/demo_config.yaml")
print(f"Database config: {config.get_section('database')}")  # Debug
# Override logging config to use project root
if 'logging' not in config._data:
    config._data['logging'] = {}
config._data['logging']['log_dir_name'] = os.path.join(project_root, 'logs')
logger = Logging(config, instance_id="demo_relations_notebook")
db_handler = DatabaseHandler(config, logger)
engine = db_handler.get_engine()

For Heroku database, set DATABASE_URL environment variable or use the line below.
Database config: {'user': 'u64g8eocr7nhul', 'password': 'p3b8ecfc9a637f11fb55f26fcf499d36f5f6a5c90a59f1c99a40fce32e4b549b7', 'host': 'ca932070ke6bv1.cluster-czrs8kj4isg7.us-east-1.rds.amazonaws.com', 'port': '5432', 'database': 'da1co8rkl4afh'}


## Cleanup

Run the cleanup script to drop the demo tables before creating new data.

In [2]:
!type ..\sqls\demo_relations_cleanup.sql | heroku pg:psql DATABASE -a seanstrader
logger.info("Cleaned up demo tables")

--> Connecting to postgresql-shaped-71623
NOTICE:  table "trade_tags" does not exist, skipping
NOTICE:  table "tags" does not exist, skipping
NOTICE:  table "trades" does not exist, skipping
NOTICE:  table "profiles" does not exist, skipping
NOTICE:  table "traders" does not exist, skipping
TradingLogger.demo_relations_notebook - INFO - Cleaned up demo tables


DROP TABLE
DROP TABLE
DROP TABLE
DROP TABLE
DROP TABLE


## Setup: Create Demo Tables

Create the required tables for the demo.

In [3]:
with engine.connect() as conn:
    # Create traders table
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS traders (
            trader_id SERIAL PRIMARY KEY,
            name VARCHAR(255) NOT NULL,
            email VARCHAR(255) NOT NULL UNIQUE
        )
    """))

    # Create profiles table
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS profiles (
            trader_id INTEGER PRIMARY KEY REFERENCES traders(trader_id) ON DELETE CASCADE,
            bio TEXT,
            risk_level VARCHAR(50)
        )
    """))

    # Create trades table
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS trades (
            trade_id SERIAL PRIMARY KEY,
            trader_id INTEGER REFERENCES traders(trader_id) ON DELETE CASCADE,
            symbol VARCHAR(50) NOT NULL,
            amount DECIMAL(10, 2) NOT NULL,
            trade_date TIMESTAMP NOT NULL
        )
    """))

    # Create tags table
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS tags (
            tag_id SERIAL PRIMARY KEY,
            name VARCHAR(50) NOT NULL UNIQUE
        )
    """))

    # Create trade_tags table
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS trade_tags (
            trade_id INTEGER REFERENCES trades(trade_id) ON DELETE CASCADE,
            tag_id INTEGER REFERENCES tags(tag_id) ON DELETE CASCADE,
            PRIMARY KEY (trade_id, tag_id)
        )
    """))

    # Insert default tags
    conn.execute(text("""
        INSERT INTO tags (name) VALUES ('Stock'), ('Long')
        ON CONFLICT (name) DO NOTHING
    """))

    conn.commit()
    logger.info("Created demo tables and inserted default tags")

TradingLogger.demo_relations_notebook - INFO - Created demo tables and inserted default tags


## Create: Add a New Trader, Profile, Trade, and Tags

Insert a new trader, their profile, a trade, and associate tags.

In [4]:
with engine.connect() as conn:
    # Insert trader
    logger.info("Creating a new trader and profile")
    trader_result = conn.execute(text("""
        INSERT INTO traders (name, email)
        VALUES (:name, :email)
        RETURNING trader_id
    """), {"name": "Eve Davis", "email": "eve@trading.com"})
    trader_id = trader_result.fetchone()[0]
    
    # Insert profile
    conn.execute(text("""
        INSERT INTO profiles (trader_id, bio, risk_level)
        VALUES (:trader_id, :bio, :risk_level)
    """), {
        "trader_id": trader_id,
        "bio": "Algo trader with a focus on futures",
        "risk_level": "High"
    })
    
    # Insert trade
    trade_result = conn.execute(text("""
        INSERT INTO trades (trader_id, symbol, amount, trade_date)
        VALUES (:trader_id, :symbol, :amount, :trade_date)
        RETURNING trade_id
    """), {
        "trader_id": trader_id,
        "symbol": "MSFT",
        "amount": 4000.00,
        "trade_date": "2025-04-14T15:00:00+00:00"
    })
    trade_id = trade_result.fetchone()[0]
    
    # Link trade to tags
    conn.execute(text("""
        INSERT INTO trade_tags (trade_id, tag_id)
        VALUES (:trade_id, (SELECT tag_id FROM tags WHERE name = 'Stock')),
               (:trade_id, (SELECT tag_id FROM tags WHERE name = 'Long'))
    """), {"trade_id": trade_id})
    
    conn.commit()
    logger.info(f"Created trader ID {trader_id} with trade ID {trade_id}")

TradingLogger.demo_relations_notebook - INFO - Creating a new trader and profile
TradingLogger.demo_relations_notebook - INFO - Created trader ID 1 with trade ID 1


## Read: Query Trader with Relations

Retrieve the trader's data, including profile, trades, and tags.

In [5]:
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT t.trader_id, t.name, p.bio, p.risk_level,
               tr.trade_id, tr.symbol, tr.amount, tr.trade_date,
               tg.name AS tag_name
        FROM traders t
        LEFT JOIN profiles p ON t.trader_id = p.trader_id
        LEFT JOIN trades tr ON t.trader_id = tr.trader_id
        LEFT JOIN trade_tags tt ON tr.trade_id = tt.trade_id
        LEFT JOIN tags tg ON tt.tag_id = tg.tag_id
        WHERE t.email = :email
    """), {"email": "eve@trading.com"})
    rows = result.fetchall()
    for row in rows:
        print(f"Trader: {row.name}, Bio: {row.bio}, Trade: {row.symbol}, Tags: {row.tag_name}")

Trader: Eve Davis, Bio: Algo trader with a focus on futures, Trade: MSFT, Tags: Stock
Trader: Eve Davis, Bio: Algo trader with a focus on futures, Trade: MSFT, Tags: Long


## Update: Modify Trader's Risk Level

Update the trader's profile to change their risk level.

In [6]:
with engine.connect() as conn:
    conn.execute(text("""
        UPDATE profiles
        SET risk_level = :risk_level
        WHERE trader_id = (SELECT trader_id FROM traders WHERE email = :email)
    """), {"risk_level": "Low", "email": "eve@trading.com"})
    conn.commit()
    logger.info("Updated Eve's risk level to Low")

TradingLogger.demo_relations_notebook - INFO - Updated Eve's risk level to Low


## Delete: Remove the Trade

Delete the trade, which cascades to `trade_tags`.

In [7]:
with engine.connect() as conn:
    conn.execute(text("""
        DELETE FROM trades
        WHERE trade_id = (SELECT trade_id FROM trades WHERE symbol = :symbol)
    """), {"symbol": "MSFT"})
    conn.commit()
    logger.info("Deleted MSFT trade")

TradingLogger.demo_relations_notebook - INFO - Deleted MSFT trade
